In [1]:
import random
import numpy as np 
import matplotlib.pyplot as plt
import csv
import os
import sys

In [2]:
POPULATION_SIZE = 50
GENOME_LENGTH = 13
MIN_VAL = 0
MAX_VAL = 10

In [3]:
def decimal_to_binary(value):
    integer = round((value - MIN_VAL) / (MAX_VAL - MIN_VAL) * (2**GENOME_LENGTH - 1))
    binary = bin(integer)[2:]
    return binary.zfill(GENOME_LENGTH)

def binary_to_decimal(bits):
    binary_str = ''.join(str(bit) for bit in bits)
    integer = int(binary_str, 2)

    return round(MIN_VAL + integer * (MAX_VAL - MIN_VAL) / (2**GENOME_LENGTH - 1), 6)

In [ ]:
def init_population(population_size):
    population = []
    for _ in range(population_size):
        # it will have 10^-3
        x = round(random.uniform(MIN_VAL, MAX_VAL), 3)
        y = round(random.uniform(MIN_VAL, MAX_VAL), 3)

        x_bin = decimal_to_binary(x)
        y_bin = decimal_to_binary(y)

        x_bits = [int(bit) for bit in x_bin]
        y_bits = [int(bit) for bit in y_bin]

        population.append(dict(x_bits=x_bits, x_decimal=x, y_bits=y_bits, y_decimal=y))
    return population

In [5]:
def fitness(population):
    # create array of population with the fitness
    population_with_fitness = []
    for chromosome in population:
        #get the x decimal and y decimal
        x = chromosome['x_decimal']
        y = chromosome['y_decimal']

        # get the binary of x and y
        x_bin = chromosome['x_bits']
        y_bin = chromosome['y_bits']

        # check if x and y fulfill the constraint, then the status will be True. If not, the status will be False. 
        status = (0 <= x <= 4) and (1 <= y <= 6) and (x + y <= 4)

        fitness_value = np.exp(-(x - 1)**2) + np.exp(-(y - 2)**2)
        if status == False: #add penalty
            if not 0 <= x <= 4:
                R = -4
                fitness_value = fitness_value + (R * (abs((x + y) - 4)**2))
                print("condition 1\n")
            if not 1 <= y <= 6:
                R = -5
                fitness_value = fitness_value + (R * (abs((x + y) - 4)**2))
                print("condition 2\n")
            if not x + y <= 4:
                R = -6
                fitness_value = fitness_value + (R * (abs((x + y) - 4)**2))
                print("condition 3\n")
        # add to array
        population_with_fitness.append(dict(x_decimal = x, x_bits = x_bin, y_decimal = y, y_bits = y_bin, fitness = fitness_value, status_constraint = status))
    
    return population_with_fitness

In [6]:
def roulette_wheel_selection(population, prev_index):
    fitness_values = [chromosome['fitness'] for chromosome in population]
    
    # shifting fitness value so the minimum is 0
    min_fitness = min(fitness_values)
    epsilon = 1e-6
    shifted_fitness = [f - min_fitness + epsilon for f in fitness_values]
    
    # make probability of each chromosome
    total_fitness = sum(shifted_fitness)
    probabilities = [f / total_fitness for f in shifted_fitness]
    
    population_with_prob = [
        {'chromosome': chromosome, 'probability': prob}
        for chromosome, prob in zip(population, probabilities)
    ]
   
    print("sorted population with probabilities: ", [
        {'chromosome': p['chromosome'], 'probability': p['probability']}
        for p in population_with_prob
    ])
    
    # select index by the probability. 
    selected_index = np.random.choice(
        len(population_with_prob), 
        p=[p['probability'] for p in population_with_prob]
    )
   
   # checking if the selected index is same as the another party. make sure it's different!
    while selected_index == prev_index:
        # if it's same, do random selection index again
        selected_index = np.random.choice(
            len(population_with_prob), 
            p=[p['probability'] for p in population_with_prob]
        )

    print("selected index: ", selected_index)
    return selected_index, population_with_prob[selected_index]['chromosome']

In [7]:
def crossover(p1, p2, crossover_rate=0.9):
    # random value to know should we do the crossover or not as long as not exceeds the crossover rate
    if random.random() > crossover_rate:
        return (p1['x_bits'], p2['x_bits'], p1['y_bits'], p2['y_bits'])

    def is_valid(x1, x2, y1, y2):
        # checking the constraint
        return (0 <= x1 <= 4 and 0 <= x2 <= 4 and
                1 <= y1 <= 6 and 1 <= y2 <= 6 and
                x1 + y1 <= 4 and x2 + y2 <= 4)

    for _ in range(10):
        point = random.randint(1, GENOME_LENGTH - 1)

        # two-point cross over
        child_x_1 = np.concatenate([p1['x_bits'][:point], p2['x_bits'][point:]])
        child_x_2 = np.concatenate([p2['x_bits'][:point], p1['x_bits'][point:]])
        child_y_1 = np.concatenate([p1['y_bits'][:point], p2['y_bits'][point:]])
        child_y_2 = np.concatenate([p2['y_bits'][:point], p1['y_bits'][point:]])

        # convert into decimal
        x1, x2 = binary_to_decimal(child_x_1), binary_to_decimal(child_x_2)
        y1, y2 = binary_to_decimal(child_y_1), binary_to_decimal(child_y_2)

        if is_valid(x1, x2, y1, y2):
            # if valid then return the offspring. 
            return (child_x_1, child_x_2, child_y_1, child_y_2)
        
    #if not, the parents will return so they won't have kids. 
    return (p1['x_bits'], p2['x_bits'], p1['y_bits'], p2['y_bits'])

In [8]:
def mutation(individual, constraint, mutation_rate=0.1):
    mutated_individual = individual.copy()

    # iterating until last genome
    for i in range(GENOME_LENGTH):
        # finding the random value. if it's fulfill the mutation rate, the mutation will be happened
        if random.random() < mutation_rate:
            mutated_individual[i] = 1 - mutated_individual[i]
    
    # change from binary to decimal
    decimal_value = binary_to_decimal(mutated_individual)

    # check the constraints
    if constraint == 'x':
        if 0 <= decimal_value <= 4:
            return mutated_individual
    elif constraint == 'y':
        if 1 <= decimal_value <= 6:
            return mutated_individual

    # return to the original individual because doesn't fulfill the constraints.
    return individual

In [ ]:
def check_status_population(population):
    count_keep = 0

    for chromosome in population:
        if chromosome['status_constraint'] == True:
            count_keep += 1
        else:
            continue
    
    # if the suitable chromosome inside population only 1 or 2, then return False because it's not ideal
    if count_keep == 0 or count_keep == 1:
        return False

    else:
        print("Safe population from checking: \n", population)
        return population

In [ ]:
def genetic_algorithm(population_size):
    population = init_population(population_size)
    population_fitness = fitness(population)

    i = 0
    generations = 100

    fitness_all = []
    x_all = []
    y_all = []
    status_all = []

    best_fitness_so_far = -np.inf
    
    while(i < generations):
        print(f"Generation {i}\n")

        result_check = check_status_population(population_fitness)

        if result_check == False:
            print(f"Init population again\n")
            population_without_fitness = init_population(population_size)
            population_fitness = fitness(population_without_fitness)
            generations += 1

            # carry forward previous best if exists
            if len(fitness_all) > 0:
                fitness_all.append(fitness_all[-1])
                x_all.append(x_all[-1])
                y_all.append(y_all[-1])
                status_all.append(status_all[-1])

            print("This generation is being skipped\n")

        else:
            new_population = []
            new_population_temp = []

            ix = -1
            ix, p1 = roulette_wheel_selection(result_check, ix)
            ix, p2 = roulette_wheel_selection(result_check, ix)

            while len(new_population) < population_size // 2:
                # Crossover
                offspring_x_1, offspring_x_2, offspring_y_1, offspring_y_2 = crossover(p1, p2)

                # Mutation
                offspring_mutation_x_1 = mutation(offspring_x_1, 'x')
                offspring_mutation_x_2 = mutation(offspring_x_2, 'x')
                offspring_mutation_y_1 = mutation(offspring_y_1, 'y')
                offspring_mutation_y_2 = mutation(offspring_y_2, 'y')

                decimal_offspring_mutation_x1 = binary_to_decimal(offspring_mutation_x_1)
                decimal_offspring_mutation_x2 = binary_to_decimal(offspring_mutation_x_2)
                decimal_offspring_mutation_y1 = binary_to_decimal(offspring_mutation_y_1)
                decimal_offspring_mutation_y2 = binary_to_decimal(offspring_mutation_y_2)

                new_child_1 = dict(x_bits=offspring_mutation_x_1, x_decimal=decimal_offspring_mutation_x1,
                                   y_bits=offspring_mutation_y_1, y_decimal=decimal_offspring_mutation_y1)
                new_child_2 = dict(x_bits=offspring_mutation_x_2, x_decimal=decimal_offspring_mutation_x2,
                                   y_bits=offspring_mutation_y_2, y_decimal=decimal_offspring_mutation_y2)

                new_population_temp = [new_child_1, new_child_2]
                new_population_temp_val = fitness(new_population_temp)
                # check again the population of the new children
                result_new = check_status_population(new_population_temp_val)

                if result_new != False:
                    new_population.append(new_child_1)
                    new_population.append(new_child_2)

                    new_population_val = fitness(new_population)
                    # check the population 
                    result_new_full = check_status_population(new_population_val)

                    iy = -1
                    iy, p1 = roulette_wheel_selection(result_new_full, iy)
                    iy, p2 = roulette_wheel_selection(result_new_full, iy)
                else:
                    print("Offspring does not satisfy the constraint, skipping\n")
                    continue

                new_population_temp.clear()
            
            population_fitness = new_population_val

            _, best_chromosome = max(enumerate(population_fitness), key=lambda x: x[1]['fitness'])
            current_best_fitness = best_chromosome['fitness']

            # Convergence logic
            if current_best_fitness >= best_fitness_so_far:
                best_fitness_so_far = current_best_fitness
                fitness_all.append(current_best_fitness)
                x_all.append(best_chromosome['x_decimal'])
                y_all.append(best_chromosome['y_decimal'])
                status_all.append(best_chromosome['status_constraint'])
                print(f"New best fitness: {current_best_fitness} at x={best_chromosome['x_decimal']}, y={best_chromosome['y_decimal']}\n")
            else:
                fitness_all.append(fitness_all[-1])
                x_all.append(x_all[-1])
                y_all.append(y_all[-1])
                status_all.append(status_all[-1])
                print(f"No improvement. Best stays at: {best_fitness_so_far}\n")

        i += 1
    
    return fitness_all, x_all, y_all, status_all

In [11]:
def do_convergence(fitness_all, x_all, y_all, status_all, filename):
    # initial values
    first_fitness = fitness_all[0]
    first_x = x_all[0]
    first_y = y_all[0]

    fitness_convergence = []
    x_convergence = []
    y_convergence = []

    # fitness convergence
    for i in range(len(fitness_all)):
        first_fitness = max(first_fitness, fitness_all[i])
        fitness_convergence.append(first_fitness)

    # x convergence
    for i in range(len(x_all)):
        first_x = max(first_x, x_all[i])
        x_convergence.append(first_x)

    # y convergence
    for i in range(len(y_all)):
        first_y = max(first_y, y_all[i])
        y_convergence.append(first_y)

    # write CSV
    csv_filename = f'ga_convergence_results_{filename}.csv'
    with open(csv_filename, mode='w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)

        writer.writerow([
            'Iteration',
            'Fitness',
            'Fitness_Convergence',
            'x',
            'x_Convergence',
            'y',
            'y_Convergence',
            'status'
        ])

        # rows
        for i in range(len(fitness_all)):
            writer.writerow([
                i + 1,
                round(fitness_all[i], 4),
                round(fitness_convergence[i], 4),
                round(x_all[i], 4),
                round(x_convergence[i], 4),
                round(y_all[i], 4),
                round(y_convergence[i], 4),
                status_all[i]
            ])

    return (
        fitness_convergence,
        x_convergence,
        y_convergence
    )

In [12]:
def make_graph(fitness_all, x_all, y_all, fitness_convergence, x_convergence, y_convergence, filename):
    fig, axs = plt.subplots(6, 1, figsize=(15, 20))
    
    axs[0].plot(fitness_all, marker='o', linestyle='-', markersize=4)
    axs[0].set_title('Fitness Over Time')
    axs[0].set_ylabel('Fitness')

    axs[1].plot(fitness_convergence, marker='o', linestyle='-', markersize=4)
    axs[1].set_title('Fitness convergence over time')
    axs[1].set_ylabel('Fitness')
    axs[1].set_xlabel('Time Steps')

    axs[2].plot(x_all, marker='o', linestyle='-', markersize=4)
    axs[2].set_title('x Over Time')
    axs[2].set_ylabel('x')

    axs[3].plot(x_convergence, marker='o', linestyle='-', markersize=4)
    axs[3].set_title('x convergence over time')
    axs[3].set_ylabel('x')
    axs[3].set_xlabel('Time Steps')

    axs[4].plot(y_all, marker='o', linestyle='-', markersize=4)
    axs[4].set_title('y Over Time')
    axs[4].set_ylabel('y')

    axs[5].plot(y_convergence, marker='o', linestyle='-', markersize=4)
    axs[5].set_title('y convergence over time')
    axs[5].set_ylabel('y')
    axs[5].set_xlabel('Time Steps')

    plt.tight_layout()
    
    save_path = f'graph_output{filename}.jpg'
    plt.savefig(save_path, format='jpg')
    plt.close(fig)

In [13]:
filename = 'result_penalty_cr09_mr01'
output_file = f'{filename}.txt'

if os.path.exists(output_file):
    os.remove(output_file)

original_stdout = sys.stdout
with open(output_file, 'w', encoding='utf-8') as f:
    sys.stdout = f
    try:
        fitness_all, x_all, y_all, status_all = genetic_algorithm(POPULATION_SIZE)
        fitness_convergence, x_convergence, y_convergence = do_convergence(fitness_all, x_all, y_all, status_all, filename)
        make_graph(fitness_all, x_all, y_all, fitness_convergence, x_convergence, y_convergence, filename)
    finally:
        sys.stdout = original_stdout